<a href="https://colab.research.google.com/github/jarl24-dev/mlops-zoomcamp/blob/main/02-dataframe-analysis/Homework2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question 1: [IPO] Withdrawn IPOs by Company Type

In [92]:
import pandas as pd
import requests
from io import StringIO

In [93]:
url='https://stockanalysis.com/ipos/withdrawn/'
headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/58.0.3029.110 Safari/537.3'
        )
    }

response = requests.get(url, headers=headers, timeout=10)
html_io = StringIO(response.text)

tables = pd.read_html(html_io)
df = tables[0]

In [94]:
df["Company Name"] = df["Company Name"].astype("string")

def companyclass(company_name):
    if "Acquisition Corp" in company_name or "Acquisition Corporation" in company_name:
        return "Acq.Corp"
    elif "Inc" in company_name or "Incorporated" in company_name:
        return "Inc"
    elif "Group" in company_name:
        return "Group"
    elif "Ltd" in company_name or "Limited" in company_name:
        return "Limited"
    elif "Holdings" in company_name:
        return "Holdings"
    else:
        return "Other"

df["Company Class"] = df["Company Name"].apply(companyclass)
df["Company Class"] = df["Company Class"].astype("string")

In [95]:
df["Price Range"] = df["Price Range"].astype("string")

def avgprice(price_range):
  price_range = price_range.split("-")
  for i in range(len(price_range)):
    try:
      price_range[i]=float(price_range[i].replace("$", "").replace(" ", ""))
    except:
      price_range[i]=0.0
  return sum(price for price in price_range) / len(price_range)

df["Avg. price"] = df["Price Range"].apply(avgprice)
df["Avg. price"] = df["Avg. price"].astype(float)

In [97]:
df["Shares Offered"] = pd.to_numeric(df["Shares Offered"], errors="coerce")

In [102]:
df["Withdrawn Value"] = df["Shares Offered"] * df["Avg. price"]

In [107]:
df.groupby("Company Class")["Withdrawn Value"].sum().sort_values(ascending=False)

,Withdrawn Value
Company Class,
Acq.Corp,4.021000e+09
Inc,2.257164e+09
Other,7.679200e+08
Limited,5.497346e+08
Holdings,7.500000e+07
Group,3.378750e+07
